In [1]:
import os
import shutil
import subprocess
import time
import re
import sys
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed


# ---------------------------------------------------------------------------
# OneDrive attribute helpers
# ---------------------------------------------------------------------------

def _attrib(flag, path):
    """
    Toggle OneDrive's pinned/online-only attribute on a file.

    shell=False here on purpose: with shell=True every call spawns BOTH
    cmd.exe and attrib.exe. attrib.exe is a real executable in
    C:\\Windows\\System32, so it can be launched directly -- that's one
    process instead of two, on every single file, thousands of times.
    """
    try:
        subprocess.run(
            ["attrib", flag, path],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        return True
    except Exception:
        return False


def force_onedrive_download(path):
    """Force OneDrive/SharePoint to download a cloud-only file."""
    return _attrib("-U", path)


def set_onedrive_online_only(path):
    """Mark a file as online-only in OneDrive/SharePoint."""
    return _attrib("+U", path)


# ---------------------------------------------------------------------------
# Copy primitives
# ---------------------------------------------------------------------------

def safe_copy(src, dst, retries=8, delay=8):
    """Copy a file, retrying if OneDrive throws a cloud-timeout error."""
    for _ in range(retries):
        try:
            shutil.copy2(src, dst)
            return True
        except OSError as e:
            msg = str(e)
            if "WinError 426" in msg or "cloud operation" in msg or "0x80070185" in msg:
                force_onedrive_download(src)
                time.sleep(delay)
                continue
            return False
    return False


def wait_for_upload_complete(path, stable_checks=2, delay=3):
    """
    Short safety buffer before re-clouding a freshly written destination file.

    Honest caveat carried over from the original: local size/mtime stop
    changing the instant shutil.copy2() returns, so this was never really
    "watching the cloud upload finish" -- it's a flat stable_checks*delay
    pause (15s by default in the original). Kept it, just shorter, and it
    now runs inside a worker thread instead of blocking the whole pipeline.
    """
    last_size = last_mtime = None
    stable_count = 0
    while stable_count < stable_checks:
        try:
            size, mtime = os.path.getsize(path), os.path.getmtime(path)
            if (size, mtime) == (last_size, last_mtime):
                stable_count += 1
            else:
                stable_count = 0
            last_size, last_mtime = size, mtime
        except OSError:
            stable_count = 0
        time.sleep(delay)


# ---------------------------------------------------------------------------
# File discovery (single pass, replaces the old count-then-walk-again)
# ---------------------------------------------------------------------------

def list_files(source_dir, filename_patterns=None):
    """One os.walk pass that returns matching files -- used for both the
    total count and the iteration, instead of walking the tree twice."""
    files = []
    for root, _, fnames in os.walk(source_dir):
        for fname in fnames:
            if filename_patterns and not any(re.search(p, fname) for p in filename_patterns):
                continue
            files.append(os.path.join(root, fname))
    return files


def count_files(source_dir, filename_patterns=None):
    """Kept for backwards compatibility with any code calling this directly."""
    return len(list_files(source_dir, filename_patterns))


# ---------------------------------------------------------------------------
# Thread-safe progress line
# ---------------------------------------------------------------------------

_print_lock = threading.Lock()


def _report(dataset_name, done, total, copied, skipped, errors):
    with _print_lock:
        percent = (done / total) * 100 if total else 100
        sys.stdout.write(
            f"\r{dataset_name} — {percent:6.2f}% "
            f"| Copied: {copied} | Skipped: {skipped} | Errors: {errors}   "
        )
        sys.stdout.flush()


# ---------------------------------------------------------------------------
# Core engine: bounded-concurrency file pipeline
# ---------------------------------------------------------------------------

def _process_dataset(source_dir, destination_drive, filename_patterns=None,
                      max_workers=5, skip_if_same_size=False, free_destination=False):
    """
    max_workers is the disk-usage knob: at most `max_workers` files will be
    fully hydrated on local disk at any moment (instead of one at a time,
    and instead of the whole dataset at once). Raise it if OneDrive and
    disk I/O keep up comfortably; lower it if OneDrive feels sluggish for
    other apps while this runs.
    """
    source_dir = os.path.abspath(source_dir)
    destination_drive = os.path.abspath(destination_drive)
    dataset_name = os.path.basename(source_dir)

    src_files = list_files(source_dir, filename_patterns)
    total_files = len(src_files)

    if total_files == 0:
        print(f"{dataset_name} — No files found.")
        return

    counters = {"checked": 0, "copied": 0, "skipped": 0, "errors": 0}
    lock = threading.Lock()

    def process_one(src_file):
        try:
            rel_path = os.path.relpath(os.path.dirname(src_file), source_dir)
            dest_dir = os.path.join(destination_drive, rel_path)
            os.makedirs(dest_dir, exist_ok=True)
            dst_file = os.path.join(dest_dir, os.path.basename(src_file))

            copy_needed = True
            if skip_if_same_size and os.path.exists(dst_file):
                try:
                    if os.path.getsize(src_file) == os.path.getsize(dst_file):
                        copy_needed = False
                except OSError:
                    pass

            copied = skipped = errored = False
            if copy_needed:
                if safe_copy(src_file, dst_file):
                    copied = True
                    set_onedrive_online_only(src_file)  # free the source side right away
                    if free_destination:
                        wait_for_upload_complete(dst_file)
                        set_onedrive_online_only(dst_file)  # free the destination side
                else:
                    errored = True
            else:
                skipped = True
        except Exception:
            copied = skipped = False
            errored = True

        with lock:
            counters["checked"] += 1
            counters["copied"] += int(copied)
            counters["skipped"] += int(skipped)
            counters["errors"] += int(errored)
            _report(dataset_name, counters["checked"], total_files,
                     counters["copied"], counters["skipped"], counters["errors"])

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(process_one, f) for f in src_files]
        for fut in as_completed(futures):
            fut.result()  # surface anything truly unexpected

    print(
        f"\n{dataset_name} — DONE | Copied: {counters['copied']} "
        f"| Skipped: {counters['skipped']} | Errors: {counters['errors']}"
    )


# ---------------------------------------------------------------------------
# Drop-in replacements for the original three functions
# (same names/behavior, now with an optional max_workers knob)
# ---------------------------------------------------------------------------

def copy_directory_to_drive(source_dir, destination_drive, filename_patterns=None, max_workers=5):
    _process_dataset(source_dir, destination_drive, filename_patterns,
                      max_workers=max_workers, skip_if_same_size=False, free_destination=False)


def update_missing_files1(source_dir, destination_drive, filename_patterns=None, max_workers=5):
    _process_dataset(source_dir, destination_drive, filename_patterns,
                      max_workers=max_workers, skip_if_same_size=True, free_destination=False)


def update_missing_files2(source_dir, destination_drive, filename_patterns=None, max_workers=5):
    _process_dataset(source_dir, destination_drive, filename_patterns,
                      max_workers=max_workers, skip_if_same_size=True, free_destination=True)

source_directory = (
    r"Y:\loading_repository\DREAM" 
)

destination_drive = r"E:\King's College London/Department of Psychosis Shared Database Initiative - Documents/Storage Repository/DREAM"


update_missing_files2(
    source_dir=source_directory,
    destination_drive=destination_drive,
    filename_patterns=None,
    max_workers=8
)

In [4]:
destination_drive = (r"Y:\storage")

source_directory = r"E:\King's College London/Department of Psychosis Shared Database Initiative - Documents/Storage Repository/"


update_missing_files1(
    source_dir=source_directory,
    destination_drive=destination_drive,
    filename_patterns=None,
    max_workers=4
)

Storage Repository — 100.00% | Copied: 241 | Skipped: 515197 | Errors: 2   
Storage Repository — DONE | Copied: 241 | Skipped: 515197 | Errors: 2
